# MO434 — Knowledge Distillation: Results & Best Students

A Comparative Study of Knowledge Distillation Schemes from Pretrained Image
Classifiers into a Lightweight ConvNet.

This notebook is the final code deliverable. It:
1. Loads the Phase-1 teachers and Phase-2/3 result tables.
2. For each **(teacher, dataset)** picks the **best student** and **reproduces**
   its test accuracy by reloading the weights and the frozen teacher classifier.
3. Reports the **parameter / GFLOPs savings** versus the teacher.
4. Surfaces the five project questions (Q1 teacher transfer, Q2 pre- vs
   post-GAP, Q3 efficiency, Q4 loss ablation, Q5 relational KD).

Run inside the `mo434-kd` container (`cd /workspace/notebooks`).

In [ ]:
import sys, json, copy
from pathlib import Path
import pandas as pd, torch

ROOT = Path.cwd().parent          # /workspace
sys.path.insert(0, str(ROOT))
from src.phase1.teachers import load_teacher_checkpoint
from src.phase1.utils import get_dataloaders, set_seed, compute_flops, count_parameters, count_total_parameters
from src.phase2.student import Student

TABLE_DIR = ROOT / "outputs" / "tables"
STUD_DIR  = ROOT / "outputs" / "students"
FIG_DIR   = ROOT / "outputs" / "figures"
DATA_ROOT = ROOT / "data"
CKPT_DIR  = ROOT / "outputs" / "checkpoints" / "teachers"
DEVICE    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BACKBONE  = {"resnet": "resnet50", "convnext": "convnext_base", "vgg": "vgg16_bn"}
CKPT_MODE = {"resnet": "finetune", "convnext": "finetune", "vgg": "frozen"}
DATASETS  = ["oxford-pets", "flowers-102"]
print("device:", DEVICE)

## 1. Teacher reference (Phase 1)
Frozen-encoder classifiers; finetune variant unfreezes the top block.

In [ ]:
teacher_cmp = pd.read_csv(TABLE_DIR / "teacher_results_comparison.csv")
teacher_cmp[["Mode","Teacher","Dataset","Test Acc (%)","Total Params (M)","GFLOPs"]]

## 2. Student results table (Phase 3)
All 72 students; `gflops` populated by `test_students.py`.

In [ ]:
students = pd.read_csv(TABLE_DIR / "student_results.csv")
students.sort_values("test_acc", ascending=False).head(10)

## 3. Best student per (teacher, dataset) — reproduce test accuracy

For each teacher we take the highest-test-accuracy student, rebuild it from its
JSON layer spec, attach the **frozen** Phase-1 classifier (deep-copied from the
teacher), load the trained weights, and re-evaluate on the test split.

In [ ]:
@torch.no_grad()
def eval_test(model, loader):
    import torch.nn.functional as F
    model.eval(); correct = n = 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        with torch.amp.autocast("cuda", dtype=torch.float16):
            logits = model(x)
        correct += (logits.argmax(1) == y).sum().item(); n += x.size(0)
    return correct / n

def cfg_by_id(dataset, sid):
    specs = json.loads((ROOT / "src" / "phase2" / f"students_{dataset}.json").read_text())
    return next(c for c in specs if c["id"] == sid)

rows = []
for ds in DATASETS:
    set_seed(291652)
    _, _, test_loader, _ = get_dataloaders(dataset_name=ds, data_root=DATA_ROOT,
                                           batch_size=128, num_workers=4)
    sub = students[students.dataset == ds]
    for tk in ["resnet", "convnext", "vgg"]:
        best = sub[sub.teacher == tk].sort_values("test_acc", ascending=False).iloc[0]
        sid  = best["id"]
        cfg  = cfg_by_id(ds, sid)
        teacher = load_teacher_checkpoint(backbone_name=BACKBONE[tk], dataset_name=ds,
                                          num_classes=cfg["num_classes"], checkpoint_dir=CKPT_DIR,
                                          mode=CKPT_MODE[tk], device=DEVICE)
        student = Student(layers=cfg["layers"], teacher_dim=cfg["teacher_dim"],
                          num_classes=cfg["num_classes"], classifier=copy.deepcopy(teacher.classifier),
                          target_mode=cfg["target_mode"], freeze_classifier=True).to(DEVICE)
        wpath = STUD_DIR / ds / f"{sid}.pth"
        student.load_state_dict(torch.load(wpath, map_location=DEVICE, weights_only=True))
        acc = eval_test(student, test_loader)
        rows.append({"dataset": ds, "teacher": tk, "best_student": sid,
                     "target_mode": cfg["target_mode"], "reproduced_test_acc": round(acc, 4),
                     "reported_test_acc": round(float(best["test_acc"]), 4),
                     "student_params": count_total_parameters(student) - count_total_parameters(student.classifier),
                     "student_gflops": round(compute_flops(student), 3)})
pd.DataFrame(rows)

## 4. Efficiency: student vs teacher (Q3)

The student saves both parameters and GFLOPs by a large factor while retaining a
substantial fraction of the teacher's accuracy.

In [ ]:
best_df = pd.DataFrame(rows)
tref = teacher_cmp.copy()
def teacher_row(tk, ds):
    r = tref[(tref.Teacher == BACKBONE[tk]) & (tref.Dataset == ds) & (tref.Mode == CKPT_MODE[tk])]
    return r.iloc[0]
eff = []
for _, r in best_df.iterrows():
    t = teacher_row(r.teacher, r.dataset)
    eff.append({**r.to_dict(),
                "teacher_params_M": round(t["Total Params (M)"], 2),
                "teacher_gflops": round(t["GFLOPs"], 2),
                "param_savings_x": round(t["Total Params (M)"]*1e6 / r["student_params"], 1),
                "gflops_savings_x": round(t["GFLOPs"] / r["student_gflops"], 1),
                "retention_%": round(100 * r["reproduced_test_acc"] / (t["Test Acc (%)"]/100), 1)})
pd.DataFrame(eff)[["dataset","teacher","best_student","reproduced_test_acc",
                   "param_savings_x","gflops_savings_x","retention_%"]]

## 5. Q1 — Which teacher transfers best?

In [ ]:
pd.read_csv(TABLE_DIR / "q1_teacher_summary.csv")

![Q1](../outputs/figures/q1_teacher_transfer.png)

## 6. Q2 — Pre-GAP feature maps vs post-GAP pooled vector

In [ ]:
pd.read_csv(TABLE_DIR / "q2_pre_vs_post.csv").head(12)

![Q2](../outputs/figures/q2_pre_vs_post_delta.png)

## 7. Q4 / Q5 — Loss function & Relational KD

Ablation on the best architecture (`arch6_6conv_res`, pre_gap) comparing:
pure MSE feature imitation, MSE+CE (main), MSE+CE+KD (softened logits),
CE+KD (logit-only Hinton baseline), and MSE+CE+RKD (relational KD, Park 2019).

In [ ]:
abl = TABLE_DIR / "q4_loss_ablation.csv"
pd.read_csv(abl) if abl.exists() else "Run: python src/phase2/analyze_ablation.py first."

![Q4](../outputs/figures/q4_loss_ablation.png)

## 8. Conclusions

- A lightweight ConvNet student (<3M params, ~10x fewer GFLOPs) recovers a large
  share of each teacher's accuracy purely by imitating its representation.
- **Q1:** retention is highest for the VGG teacher; absolute accuracy tracks the
  teacher's own accuracy (ConvNeXt strongest). *Caveat:* ResNet/ConvNeXt students
  distil from finetuned teachers, VGG from frozen.
- **Q2:** predicting the **pre-GAP** feature map beats the post-GAP vector for
  small students; the gap shrinks as student capacity grows.
- **Q3:** see Section 4 — order-of-magnitude param & FLOP savings.
- **Q4/Q5:** see Section 7 — combining MSE with CE helps; adding softened-logit
  KD and relational KD (RKD) are compared against the feature-only baseline.